![image.png](https://i.imgur.com/4fN73lZ.png)

# RLHF from Scratch: Learn a Reward, Optimize It with GRPO, and Watch It Hack

Every method in Days 1--8 assumed a reward we could **write down**. But how do you
score *"be a helpful, honest assistant"*? There is no formula. **RLHF** (Reinforcement
Learning from Human Feedback) solves this by *learning* the reward from human
**comparisons**, then optimizing a policy against it.

This lab builds the RLHF back-end end to end, at a scale that runs in **seconds on a
CPU**:

1. **A world we cannot score directly.** Each prompt has several candidate responses,
   each with a *hidden* true quality (the **gold reward**) we are not allowed to read.
2. **Stage 2 -- the reward model.** Collect noisy human **preferences** and fit a
   **Bradley--Terry** reward model $r_\phi$ to them. It is only a *proxy* for the gold
   reward. **(TASK 1)**
3. **Stage 3 -- GRPO.** Optimize the policy against $r_\phi$ with **GRPO** (the
   critic-free method from Day 4) under a **KL-to-reference leash**. **(TASK 2)**
4. **The punchline -- reward hacking.** Optimize the proxy too hard (drop the leash)
   and the policy climbs the proxy while the **gold reward collapses** -- Goodhart's
   law. The KL leash is exactly what holds it back.

We optimize with **GRPO**, not PPO: it is the modern RLHF back-end (DeepSeek-R1) and
you already built it from scratch on Day 4, so here it is *given* and we focus on the
two RLHF-specific pieces -- the reward-model loss and the KL leash.

## Setup

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" torch numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

device = torch.device("cpu")
torch.manual_seed(0)
np.random.seed(0)
print("device:", device)

config = {
    "num_prompts":      6,
    "num_responses":    12,    # candidate responses per prompt
    "pairs_per_prompt": 60,    # preference budget -> RM is a decent but imperfect proxy
    "sft_alpha":        1.5,   # how gold-aware the SFT reference already is
    "grpo_iters":       150,
    "group_size":       16,    # G responses sampled per prompt (GRPO group)
    "clip_eps":         0.2,   # PPO/GRPO clip range
    "lr":               0.05,
    "inner_epochs":     4,     # gradient steps per sampled group
    "betas":            [0.0, 2.0, 10.0],   # KL leash: none / healthy / too much
}

## The Setup, Concretely

Before any code, here is exactly what we simulate, and why.

**The data.** There are `num_prompts = 6` prompts (think: six questions put to a
chatbot). Each prompt has `num_responses = 12` candidate responses. We do **not** model
the *words* of a response -- a response is just an index `0..11`. What matters is that
every `(prompt, response)` pair has one hidden number:

> `gold_quality[prompt, response]` = **how much a human would actually like that
> response.**

So the entire "world" is a `6 x 12` table of numbers. **The learner never sees this
table.** It exists only so that we, the lab authors, can grade the policy at the end and
catch it cheating.

**Why abstract away the text?** Because the RLHF mechanism -- comparisons $\to$ reward
model $\to$ policy update $\to$ reward hacking -- has nothing to do with language.
Swapping the language model for a `6 x 12` lookup table lets the whole pipeline run in
about a second and makes the failure impossible to miss. (The **DPO lab** puts a real
language model back in.)

**The policy.** $\pi_\theta(\text{response} \mid \text{prompt})$ is a softmax over the 12
responses of a prompt. "Acting" means picking **one response for one prompt** -- a single
step. There is no environment, no episode, no next state. The policy **starts as**
$\pi_{\text{ref}}$, the **SFT reference**: a policy that already leans toward good
responses (like a model trained on good demonstrations) but is far from optimal.

**What we are trying to achieve.** Using *only* pairwise human comparisons -- never the
gold numbers -- move the policy's probability mass onto high-gold responses and **beat
the SFT reference**. Then watch that goal *fail* when we optimize too hard.

**The three numbers to keep your eye on:**

| number | who sees it | what it means |
|---|---|---|
| **gold** reward | only us, for grading | what we actually want |
| **proxy** reward | the optimizer | the reward model's guess at gold |
| **SFT** gold | baseline | what we started from |

In [ ]:
def build_world(cfg):
    """The hidden gold-quality table, plus the SFT reference policy built from it."""
    gold = torch.randn(cfg["num_prompts"], cfg["num_responses"])
    # SFT reference: already leans toward high-gold responses, but is far from optimal.
    reference_logits = cfg["sft_alpha"] * gold
    return gold, reference_logits


gold_quality, reference_logits = build_world(config)
reference_policy = torch.softmax(reference_logits, dim=1)

print(f"world: {config['num_prompts']} prompts x {config['num_responses']} candidate responses")
print(f"gold_quality is a {tuple(gold_quality.shape)} table -- HIDDEN from the learner\n")

# Grading numbers. The learner never uses these; we print them so we can judge it later.
sft_gold = (reference_policy * gold_quality).sum(dim=1).mean().item()
best_possible = gold_quality.max(dim=1).values.mean().item()
print(f"SFT reference gold reward : {sft_gold:.3f}   <- what we start from")
print(f"best possible gold reward : {best_possible:.3f}   <- always pick each prompt's best response")
print("\nGoal: beat the SFT number using ONLY pairwise human comparisons.")

# Peek at one prompt: the hidden quality of each of its 12 candidate responses.
plt.figure(figsize=(7, 3))
plt.bar(range(config["num_responses"]), gold_quality[0].numpy(), color="tab:green")
plt.xlabel("candidate response index")
plt.ylabel("hidden gold quality")
plt.title("Prompt 0: true quality of each response (the learner cannot see this)")
plt.tight_layout()
plt.show()

## Stage 2a: Collect Human Preferences

People are bad at giving a *number* but good at **comparing**. So for each prompt we
show a human two candidate responses and they pick the better one. We simulate that
with the **Bradley--Terry** model: the chance a human prefers response $i$ over $j$
grows with their gold-quality gap,
$$P(i \succ j) = \sigma\big(g_i - g_j\big).$$
The preference budget is deliberately **small**, so the reward model we fit next will
be a *decent but imperfect* proxy -- which is exactly what makes reward hacking
possible.

In [ ]:
def collect_preferences(gold, cfg):
    """Simulate noisy human comparisons using Bradley-Terry on the hidden gold quality.

    Each record is (prompt, winner_response, loser_response) -- indices only.
    A human shown two responses picks the better one, but noisily: the closer the two
    qualities are, the more likely the human picks the worse one.
    """
    preferences = []
    for prompt in range(cfg["num_prompts"]):
        for _ in range(cfg["pairs_per_prompt"]):
            first, second = np.random.choice(cfg["num_responses"], size=2, replace=False)
            quality_gap = gold[prompt, first] - gold[prompt, second]
            prob_first_preferred = torch.sigmoid(quality_gap).item()
            if np.random.random() < prob_first_preferred:
                winner, loser = first, second
            else:
                winner, loser = second, first
            preferences.append((prompt, int(winner), int(loser)))
    return preferences


preferences = collect_preferences(gold_quality, config)
print(f"collected {len(preferences)} pairwise preferences "
      f"({config['pairs_per_prompt']} per prompt)")
print(f"one record looks like (prompt, winner, loser): {preferences[0]}")
print("\nThe gold table was used ONLY to sample these winners. From here on, the")
print("learner sees nothing but this list of comparisons.")

## Stage 2b: Fit the Reward Model (Bradley--Terry) &mdash; TASK 1

The reward model $r_\phi(\text{prompt}, \text{response})$ outputs one number per
response. We train it so the **preferred** response scores above the **rejected** one,
by maximum likelihood under the same Bradley--Terry model. The negative log-likelihood
of a single comparison $(y^+ \succ y^-)$ is
$$L_{\text{RM}} = -\log \sigma\big(r_\phi(y^+) - r_\phi(y^-)\big).$$
Only the *gap* between scores matters; the absolute scale is free. **Remember this
$-\log\sigma(\Delta)$ shape -- it returns in the DPO lab.**

In [ ]:
def train_reward_model(preferences, cfg, steps=300, lr=0.1):
    """Fit a reward table r_phi by Bradley-Terry maximum likelihood on the preferences.

    r_phi[prompt, response] is one learned score per candidate response. It is the
    learner's PROXY for the hidden gold quality.
    """
    reward = torch.zeros(cfg["num_prompts"], cfg["num_responses"], requires_grad=True)
    optimizer = torch.optim.Adam([reward], lr=lr)

    prompt_index = torch.tensor([prompt for prompt, winner, loser in preferences])
    winner_index = torch.tensor([winner for prompt, winner, loser in preferences])
    loser_index  = torch.tensor([loser  for prompt, winner, loser in preferences])

    for _ in range(steps):
        reward_winner = reward[prompt_index, winner_index]
        reward_loser  = reward[prompt_index, loser_index]
        # TASK 1: the Bradley-Terry reward-model loss. Under Bradley-Terry the chance a
        #   human prefers the winner is sigmoid(reward_winner - reward_loser). Turn that
        #   into the negative log-likelihood of the observed comparisons, averaged over
        #   the batch. (Recall the -log sigmoid(gap) shape from the notes.)
        loss = None
        if loss is None:
            raise NotImplementedError("TASK 1: Bradley-Terry reward-model loss")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return reward.detach()


reward_model = train_reward_model(preferences, config)

### How good is our proxy?

The reward model was trained only on a handful of noisy comparisons, so it is an
**imperfect** stand-in for the gold reward: it ranks the *best* response correctly on
only some prompts, and it disagrees with gold elsewhere. That gap is the crack that
reward hacking will pry open.

In [ ]:
# The reward model is a PROXY for gold. Two ways to see how imperfect it is:
rm_top = reward_model.argmax(dim=1)
gold_top = gold_quality.argmax(dim=1)
top_match = (rm_top == gold_top).float().mean().item()
print(f"RM picks the true-best response on {top_match:.0%} of prompts")

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
# left: agreement in rankings, all (prompt, response) pairs
left.scatter(gold_quality.flatten().numpy(), reward_model.flatten().numpy(),
             alpha=0.6, color="tab:blue")
left.set_xlabel("gold quality (true)")
left.set_ylabel("reward-model score (proxy)")
left.set_title(f"Proxy vs gold  (correlated but imperfect)")
left.grid(alpha=0.3)
# right: one prompt, gold vs learned RM, side by side (z-scored so scales are comparable)
def zscore(row):
    return (row - row.mean()) / (row.std() + 1e-8)
width = 0.4
positions = np.arange(config["num_responses"])
right.bar(positions - width / 2, zscore(gold_quality[0]).numpy(), width,
          label="gold", color="tab:green")
right.bar(positions + width / 2, zscore(reward_model[0]).numpy(), width,
          label="reward model", color="tab:blue")
right.set_xlabel("candidate response")
right.set_ylabel("score (z-scored)")
right.set_title("Prompt 0: where the proxy disagrees with gold")
right.legend()
right.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Stage 3: Optimize the Policy with GRPO + the KL Leash &mdash; TASK 2

Now the RL step: move $\pi_\theta$ toward responses the **reward model** likes, starting
from the SFT reference. We use **GRPO** (Day 4), the critic-free method behind modern
LLM RL.

**What a "group" means here.** For each prompt we sample `group_size = 16` responses
from the current policy (with replacement -- the same response can come up twice) and
score all 16 with the reward model. Their **mean and std become the baseline**, so we
never train a critic:
$$\hat A_i = \frac{r_i - \operatorname{mean}(r_{1..G})}{\operatorname{std}(r_{1..G})}$$
A positive $\hat A_i$ means "this response beat its group average -- make it more
likely." We then update with the same **clipped probability ratio** as PPO.

All of the above is **given** (Day-4 machinery). The one new, RLHF-specific piece is the
**leash**: subtract $\beta\,D_{\mathrm{KL}}(\pi_\theta \,\|\, \pi_{\text{ref}})$ so the
policy cannot wander far from the trusted SFT model. That KL term is **TASK 2**.

$$J = \underbrace{\mathbb{E}\big[\min(\rho\hat A,\ \operatorname{clip}(\rho,1{-}\varepsilon,1{+}\varepsilon)\hat A)\big]}_{\text{given (Day 4)}}
\;-\; \beta\, \underbrace{D_{\mathrm{KL}}(\pi_\theta \,\|\, \pi_{\text{ref}})}_{\textbf{TASK 2}}$$

Larger $\beta$ = shorter leash = the policy stays closer to SFT.

In [ ]:
def grpo_optimize(reward_model, reference_logits, gold, beta, cfg):
    """GRPO against the reward model, leashed to the SFT reference by beta * KL.

    Returns, per iteration, the gold and proxy reward the policy earns, plus the final
    policy. Everything except the KL leash is Day-4 GRPO machinery.
    """
    policy_logits = reference_logits.clone().requires_grad_(True)   # start from SFT
    optimizer = torch.optim.Adam([policy_logits], lr=cfg["lr"])
    reference_logprob = torch.log_softmax(reference_logits, dim=1)

    gold_history, proxy_history = [], []
    for _ in range(cfg["grpo_iters"]):
        with torch.no_grad():
            policy_probs = torch.softmax(policy_logits, dim=1)
            # sample a GROUP of responses per prompt, and score them with the reward model
            sampled_responses = torch.multinomial(policy_probs, cfg["group_size"], replacement=True)
            old_logprob = torch.log_softmax(policy_logits, dim=1)
            rewards = reward_model.gather(1, sampled_responses)      # (num_prompts, group_size)
            # group-relative advantage: no critic, just the group's own mean and std (Day 4)
            group_mean = rewards.mean(dim=1, keepdim=True)
            group_std = rewards.std(dim=1, keepdim=True)
            advantage = (rewards - group_mean) / (group_std + 1e-6)

        for _ in range(cfg["inner_epochs"]):
            logprob = torch.log_softmax(policy_logits, dim=1)
            ratio = torch.exp(logprob.gather(1, sampled_responses)
                              - old_logprob.gather(1, sampled_responses))
            clipped_ratio = torch.clamp(ratio, 1 - cfg["clip_eps"], 1 + cfg["clip_eps"])
            surrogate = torch.min(ratio * advantage, clipped_ratio * advantage).mean()

            # TASK 2: the KL-to-reference leash. Measure how far the current policy has
            #   drifted from the SFT reference: the KL divergence of the policy from the
            #   reference for each prompt, averaged over prompts. Adding beta * (that
            #   leash) to the loss is what stops the policy chasing the proxy.
            #   (policy_probs and policy_logprob are below; reference_logprob = log pi_ref.)
            policy_probs = torch.softmax(policy_logits, dim=1)
            policy_logprob = torch.log_softmax(policy_logits, dim=1)
            kl_to_reference = None
            if kl_to_reference is None:
                raise NotImplementedError("TASK 2: KL-to-reference leash")
            loss = -surrogate + beta * kl_to_reference

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            policy_probs = torch.softmax(policy_logits, dim=1)
            gold_history.append((policy_probs * gold).sum(dim=1).mean().item())
            proxy_history.append((policy_probs * reward_model).sum(dim=1).mean().item())
    final_policy = torch.softmax(policy_logits, dim=1).detach()
    return np.array(gold_history), np.array(proxy_history), final_policy

## The Payoff: Reward Hacking

The reward model is a **proxy**. Here is what happens when we optimize it at three
leash strengths and watch **both** rewards:

- the **proxy** reward (the RM score the policy earns -- what the optimizer sees), and
- the **gold** reward (the true quality -- what we actually want, but the optimizer
  never sees).

This is exactly how reward hacking is detected in practice: re-score the policy with a
trusted *gold* reward model and look for the two curves diverging (Gao et al., 2022).
Here we *own* the gold reward, so the divergence is exact.

In [ ]:
runs = {}
for beta in config["betas"]:
    gold_history, proxy_history, final_probs = grpo_optimize(
        reward_model, reference_logits, gold_quality, beta, config)
    runs[beta] = {"gold": gold_history, "proxy": proxy_history, "probs": final_probs}
    print(f"beta={beta:5}:  proxy {proxy_history[0]:.2f} -> {proxy_history[-1]:.2f}   "
          f"gold {gold_history[0]:.2f} -> peak {gold_history.max():.2f} -> end {gold_history[-1]:.2f}")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {0.0: "tab:red", 2.0: "tab:green", 10.0: "tab:gray"}
labels = {0.0: "beta=0  (no leash)", 2.0: "beta=2  (healthy leash)", 10.0: "beta=10  (over-leashed)"}

# left: the proxy reward the optimizer sees -- it just keeps climbing when unleashed
for beta in config["betas"]:
    left.plot(runs[beta]["proxy"], color=colors[beta], label=labels[beta])
left.set_xlabel("GRPO iteration")
left.set_ylabel("proxy reward (what the optimizer sees)")
left.set_title("Proxy reward: more optimization = higher")
left.legend()
left.grid(alpha=0.3)

# right: the gold reward we actually want -- beta=0 peaks then FALLS (reward hacking)
for beta in config["betas"]:
    right.plot(runs[beta]["gold"], color=colors[beta], label=labels[beta])
right.axhline(sft_gold, ls="--", color="black", lw=1, label="SFT reference")
right.set_xlabel("GRPO iteration")
right.set_ylabel("gold reward (what we truly want)")
right.set_title("Gold reward: no leash peaks, then hacks downward")
right.legend()
right.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Goodhart in one line: beta=0 earns the HIGHEST proxy but a LOWER gold than the "
      "leashed run.\nThe reward model was only a proxy; optimizing it too hard "
      "diverges from what we wanted.")

In [ ]:
# Where did each policy put its probability mass on prompt 0?
prompt = 0
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
panels = [("SFT reference", reference_policy[prompt].numpy(), "tab:blue"),
          ("beta=0 (hacked)", runs[0.0]["probs"][prompt].numpy(), "tab:red"),
          ("beta=2 (leashed)", runs[2.0]["probs"][prompt].numpy(), "tab:green")]
true_best = int(gold_quality[prompt].argmax())
proxy_best = int(reward_model[prompt].argmax())
for axis, (title, probs, color) in zip(axes, panels):
    bars = axis.bar(range(config["num_responses"]), probs, color=color)
    bars[true_best].set_edgecolor("black")
    bars[true_best].set_linewidth(2.5)
    axis.set_title(title)
    axis.set_xlabel("response")
    axis.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("policy probability")
plt.suptitle(f"Prompt 0 -- black outline = true-best response (#{true_best}); "
             f"the proxy's favourite is #{proxy_best}")
plt.tight_layout()
plt.show()

## Takeaways

- **RLHF learns the reward it cannot write down.** Humans compare; a **Bradley--Terry**
  reward model turns those comparisons into a score; GRPO optimizes the policy against
  it. That is the whole SFT $\to$ RM $\to$ GRPO pipeline.
- **The reward model is only a proxy.** Optimize it too hard (`beta = 0`) and the
  policy climbs the **proxy** while the **gold** reward peaks and then *falls* --
  **Goodhart's law / reward hacking**. Nothing is broken; the proxy was simply not the
  true objective.
- **The KL leash is the fix.** A healthy `beta` keeps the policy near the trusted SFT
  reference, capturing most of the real improvement while refusing to chase the proxy
  off a cliff. Too large a `beta` and the policy never moves. The **sign and size of
  that leash is the whole story.**
- Same optimizer as Day 4 (**GRPO**), same $-\log\sigma(\Delta)$ preference loss that
  returns in the **DPO** lab.

## Credits & Further Reading

The ideas implemented here, with the original sources:

- **Learning a reward from comparisons** &mdash; Christiano, Leike, Brown, Martic, Legg,
  Amodei, *Deep Reinforcement Learning from Human Preferences*, NeurIPS 2017.
  [arxiv.org/abs/1706.03741](https://arxiv.org/abs/1706.03741)
- **The RLHF pipeline for LLMs (SFT $\to$ RM $\to$ RL)** &mdash; Ouyang et al.,
  *Training language models to follow instructions with human feedback* (InstructGPT),
  2022. [arxiv.org/abs/2203.02155](https://arxiv.org/abs/2203.02155)
- **The Bradley--Terry preference model** &mdash; Bradley & Terry, *Rank Analysis of
  Incomplete Block Designs*, Biometrika, 1952.
- **Reward-model over-optimization / the gold-vs-proxy curve** &mdash; Gao, Schulman,
  Hilton, *Scaling Laws for Reward Model Overoptimization*, 2022.
  [arxiv.org/abs/2210.10760](https://arxiv.org/abs/2210.10760)
- **GRPO (critic-free group-relative optimization)** &mdash; Shao et al.,
  *DeepSeekMath*, 2024 [arxiv.org/abs/2402.03300](https://arxiv.org/abs/2402.03300);
  DeepSeek-AI, *DeepSeek-R1*, 2025
  [arxiv.org/abs/2501.12948](https://arxiv.org/abs/2501.12948).
- **Toy-scale RLHF framing** that inspired this lab's structure &mdash; MetricGate,
  *Reinforcement Learning from Human Feedback*,
  [metricgate.com/blogs/rlhf-reinforcement-learning-human-feedback](https://metricgate.com/blogs/rlhf-reinforcement-learning-human-feedback/).